# quant-retrieval on a free GPU

Runs training and evaluation on Colab's T4 instead of a laptop. Everything here
is a thin wrapper: the logic lives in `scripts/` and `src/` in the repo, and this
notebook only clones, restores two artifacts, and runs commands. Nothing is
implemented here, on purpose, so the cloud and the laptop run identical code.

## One-time setup, before the first run

Put these two things in a folder called `quant-retrieval` at the top level of
your Google Drive:

    quant-retrieval/
      minilm_tuned_epoch3/     <- the whole folder from checkpoints/minilm_tuned/epoch-3
      negatives.parquet        <- from data/processed/negatives.parquet

That is about 88 MB and you only do it once. Everything else, including the
26,152 document corpus, gets rebuilt here in about a minute, because the data
pipeline is deterministic and runs on CPU.

The tuned checkpoint is uploaded rather than retrained because it was trained on
the laptop's GPU. Retraining it here would produce slightly different weights and
every number already committed would stop being comparable.

## Each run

Runtime, Change runtime type, T4 GPU. Then Runtime, Run all. The last cell prints
the result files to copy back into the repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print("NO GPU: set Runtime > Change runtime type > T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ARTIFACTS = Path('/content/drive/MyDrive/quant-retrieval')
assert ARTIFACTS.exists(), f"create {ARTIFACTS} in your Drive first, see the notes above"
print(sorted(p.name for p in ARTIFACTS.iterdir()))

In [ ]:
%cd /content
!rm -rf quant-retrieval
!git clone -q https://github.com/melihgiray/quant-retrieval.git
%cd /content/quant-retrieval

# Colab already ships torch built for this GPU. Installing our pinned version
# would replace it with a slower or broken build, so install the package without
# its dependencies and add only what Colab lacks.
# faiss is optional in pyproject because it cannot share a process with
# torch on macOS. On Linux they coexist, which is why the ANN work runs here.
!pip install -q py7zr faiss-cpu
!pip install -q -e . --no-deps

import importlib
missing = [m for m in ("pandas","numpy","scipy","torch","transformers","yaml","bs4","lxml","tqdm","pyarrow")
           if not importlib.util.find_spec(m)]
print("missing:", missing or "nothing")
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Rebuild the dataset. Deterministic and CPU only, so it matches the laptop byte
# for byte. About a minute.
!python scripts/download_data.py
!python scripts/build_dataset.py 2>&1 | tail -5

In [ ]:
# Restore the two things that cannot be rebuilt here.
!mkdir -p checkpoints/minilm_tuned
!cp -r "/content/drive/MyDrive/quant-retrieval/minilm_tuned_epoch3" checkpoints/minilm_tuned/epoch-3
!cp "/content/drive/MyDrive/quant-retrieval/negatives.parquet" data/processed/negatives.parquet
!ls checkpoints/minilm_tuned/epoch-3 && ls -la data/processed/negatives.parquet

## What to run

Edit this list. Each entry is a shell command run in order, and the run stops at
the first failure so a broken step does not look like a clean sweep.

In [ ]:
COMMANDS = [
    # 1. Train the reranker on a mix of mined and random negatives. ~15 min.
    "python scripts/train_reranker.py --config configs/reranker_mixed.yaml",

    # 2. Probe it against random documents BEFORE the expensive evaluations.
    #    The broken reranker scored 43% on validation and 28% on training
    #    questions against four random documents, where chance is 20%. A working
    #    one should be far above 90% on both. If this is near chance, the
    #    pipeline numbers below are already decided.
    "python scripts/probe_reranker.py --checkpoint checkpoints/reranker_mixed/epoch-2",

    # 3. The three pipelines with the new reranker.
    "python scripts/evaluate.py --config configs/bm25_rerank_mixed.yaml",
    "python scripts/evaluate.py --config configs/dense_rerank_mixed.yaml",
    "python scripts/evaluate.py --config configs/hybrid_rerank_mixed.yaml",

    # 4. Re-run every other evaluation. Results now carry per-question scores,
    #    which significance testing needs and the older files do not have.
    #    Metrics must come back identical; anything else means something broke.
    *[
        f"python scripts/evaluate.py --config configs/{name}.yaml"
        for name in [
            "bm25", "minilm_frozen",
            "minilm_tuned_epoch1", "minilm_tuned_epoch2", "minilm_tuned_epoch3",
            "minilm_batch16_epoch3", "minilm_batch32_epoch3", "minilm_batch128_epoch3",
            "minilm_hardneg_epoch1", "minilm_hardneg_epoch2", "minilm_hardneg_epoch3",
            "minilm_cls_epoch3", "hybrid",
            "bm25_rerank", "dense_rerank", "hybrid_rerank",
        ]
    ],

    # 5. The ANN tests, which are skipped on the laptop because faiss and torch
    #    cannot share a process there. This is the machine that can run them.
    "pytest -q tests/test_ann.py",

    # 6. Embed the corpus once. Everything downstream reads these vectors
    #    instead of recomputing them, including the ANN sweep and the demo.
    "python scripts/export_index.py --checkpoint checkpoints/minilm_tuned/epoch-3",

    # 7. Where does hybrid's 60ms actually go. Cheap, and it needs the index
    #    built, so it belongs in the same session.
    "python scripts/profile_pipeline.py --config configs/hybrid.yaml --queries 100",
    "python scripts/profile_pipeline.py --config configs/minilm_tuned_epoch3.yaml --queries 100",

    # 8. Significance tests on the comparisons the write-up leans on. CPU only.
    *[
        f"python scripts/compare_runs.py --baseline results/{a}_val.json"
        f" --candidate results/{b}_val.json --metric {metric}"
        for a, b, metric in [
            ("minilm_frozen", "minilm_tuned_epoch3", "ndcg_at_10"),
            ("minilm_tuned_epoch3", "hybrid", "ndcg_at_10"),
            ("minilm_tuned_epoch3", "hybrid", "recall_at_100"),
            ("minilm_tuned_epoch3", "minilm_hardneg_epoch3", "ndcg_at_10"),
            ("minilm_tuned_epoch3", "minilm_hardneg_epoch3", "recall_at_100"),
            ("minilm_tuned_epoch3", "minilm_cls_epoch3", "ndcg_at_10"),
            ("minilm_batch32_epoch3", "minilm_tuned_epoch3", "ndcg_at_10"),
            ("bm25", "minilm_frozen", "ndcg_at_10"),
        ]
    ],
]

import subprocess, sys, time
failed = []
for command in COMMANDS:
    print("=" * 70, "\n", command, flush=True)
    started = time.time()
    completed = subprocess.run(command, shell=True)
    print(f"[{time.time() - started:.0f}s, exit {completed.returncode}]", flush=True)
    if completed.returncode != 0:
        failed.append(command)
        if "train_reranker" in command or "evaluate.py" in command:
            sys.exit(f"stopped, later steps depend on this: {command}")
print("\nALL DONE" + (f", {len(failed)} non-critical failures: {failed}" if failed else ""))

## Optional second run: the ANN scaling study

Do not run this in the same session as the block above. It downloads about 1.3GB
of extra Stack Exchange dumps, unpacks them, embeds roughly 400,000 documents and
then sweeps the index, which is another half hour or so on top.

To run it, set `SCALING = True` in the next cell and Run all. The main COMMANDS
block is skipped when it is on.

In [ ]:
SCALING = False   # set True for the scaling study instead of the main run

SCALING_COMMANDS = [
    # Extra corpora. Sizes nest, so 100k is a prefix of 400k and a change
    # between them is the size rather than a different sample.
    "python scripts/build_scaling_corpus.py --sizes 100000 400000",

    # Embed each one. The quant corpus is already done by the main run.
    "python scripts/export_index.py --corpus artifacts/scaling_corpus_100000.parquet"
    " --out artifacts/scale_100000",
    "python scripts/export_index.py --corpus artifacts/scaling_corpus_400000.parquet"
    " --out artifacts/scale_400000",

    # Sweep ef_search at every size and measure recall against exact search.
    "python scripts/ann_sweep.py --embeddings artifacts artifacts/scale_100000"
    " artifacts/scale_400000",
]

if SCALING:
    import subprocess, sys, time
    for command in SCALING_COMMANDS:
        print("=" * 70, "\n", command, flush=True)
        started = time.time()
        completed = subprocess.run(command, shell=True)
        print(f"[{time.time() - started:.0f}s, exit {completed.returncode}]", flush=True)
        if completed.returncode != 0:
            sys.exit(f"stopped: {command}")
    print("\nSCALING DONE")
else:
    print("SCALING is off, the main run above is what executed")

In [ ]:
# Zip the results and drop them in Drive. Twenty JSON files is too many to
# paste back one at a time; download the zip and unpack it into the repo.
!mkdir -p "/content/drive/MyDrive/quant-retrieval/out"
!cd /content/quant-retrieval && zip -qr /content/results.zip results
!cp /content/results.zip "/content/drive/MyDrive/quant-retrieval/out/results.zip"
!cp -r /content/quant-retrieval/checkpoints "/content/drive/MyDrive/quant-retrieval/out/" 2>/dev/null || true
!ls -la "/content/drive/MyDrive/quant-retrieval/out/results.zip"

from google.colab import files
files.download("/content/results.zip")   # also offers it as a direct download
print("\nUnzip this over the repo folder, replacing results/, then the numbers can be committed.")

In [ ]:
# The single number that decides whether the fix worked.
import json, pathlib
probe = pathlib.Path("results/reranker_mixed_probe.json")
if probe.exists():
    report = json.loads(probe.read_text())
    for entry in report["splits"]:
        verdict = "GOOD" if entry["top_one_accuracy"] > 0.9 else "STILL BROKEN"
        print(f"{entry['split']:>5}: {entry['top_one_accuracy']:.1%} "
              f"(chance {entry['chance']:.0%})  {verdict}")
    print("\nFor comparison, the reranker trained on mined negatives alone scored")
    print("43% on validation and 28% on training questions.")
else:
    print("no probe output, the training step probably failed")